<a href="https://colab.research.google.com/github/ranimiftah12345678/ML_22_Ranimiftah/blob/main/JS06/JS06_TUGAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. SVM Pada voice.csv

1. Load dan Cek Data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("voice.csv")
print(df.shape)
print(df.columns.tolist())
print(df.iloc[:, -1].value_counts())   # kolom terakhir = label
df.head()

(3168, 21)
['meanfreq', 'sd', 'median', 'Q25', 'Q75', 'IQR', 'skew', 'kurt', 'sp.ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx', 'label']
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


2. Menyiapkan Fitur dan Label

In [2]:
from sklearn.preprocessing import LabelEncoder

X = df.iloc[:, :-1].values
y = LabelEncoder().fit_transform(df.iloc[:, -1])

3. Evaluasi semua Kombinasi Split X Kernel

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

splits = {"70:30": 0.30, "80:20": 0.20}
kernels = ["linear", "poly", "rbf"]

rows = []
for split_name, test_size in splits.items():
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    for k in kernels:
        model = make_pipeline(StandardScaler(), SVC(kernel=k))
        model.fit(Xtr, ytr)

        acc_train = accuracy_score(ytr, model.predict(Xtr))
        acc_test = accuracy_score(yte, model.predict(Xte))
        rows.append([split_name, k, acc_train, acc_test])

hasil1 = pd.DataFrame(rows, columns=["Split", "Kernel", "Akurasi Train", "Akurasi Test"])
hasil1

,Split,Kernel,Akurasi Train,Akurasi Test
0,70:30,linear,0.972034,0.978970
1,70:30,poly,0.966622,0.960042
2,70:30,rbf,0.983762,0.983176
3,80:20,linear,0.976717,0.974763
4,80:20,poly,0.968035,0.955836
5,80:20,rbf,0.983820,0.982650


4. Tabel Ringkas

In [4]:
tabel = hasil1.pivot(index="Kernel", columns="Split", values="Akurasi Test")
tabel = tabel[["70:30", "80:20"]].round(4)
tabel

Split,70:30,80:20
Kernel,,
linear,0.9790,0.9748
poly,0.9600,0.9558
rbf,0.9832,0.9826


#2. Klasifikasi Siang/Malam dengan SVM RBF + Fitur Histogram

1. Load Gambar

In [9]:
!unzip -q images.zip -d dataset

In [10]:
from pathlib import Path
import matplotlib.image as mpimg
import cv2

train_dir = "dataset/images/training/"
test_dir = "dataset/images/test/"

def load_dataset(img_dir):
    img_list = []
    for d in Path(img_dir).glob('*'):
        if not d.is_dir():
            continue
        label = d.name          # 'day' atau 'night'
        for file in d.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

all_img = load_dataset(train_dir) + load_dataset(test_dir)
print(f'Total gambar: {len(all_img)}')   # diharapkan 400

Total gambar: 400


2. Prapengolahan Data

In [11]:
def preprocess(img_list):
    out = []
    for img, label in img_list:
        std_img = cv2.resize(img, (1100, 600))
        out.append((std_img, 1 if label == 'day' else 0))   # day=1, night=0
    return out

all_std = preprocess(all_img)

3. Ekstraksi Fitur Histogram

In [12]:
def hist_feature(image, bins=16):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    ranges = [[0, 180], [0, 256], [0, 256]]   # rentang H, S, V di OpenCV
    feats = []
    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [bins], ranges[ch]).flatten()
        h = h / h.sum()                        # normalisasi
        feats.extend(h)
    return np.array(feats)

X2 = np.array([hist_feature(img) for img, _ in all_std])
y2 = np.array([lbl for _, lbl in all_std])
print(X2.shape, y2.shape)    # diharapkan (400, 48) (400,)

(400, 48) (400,)


4. Split 80:20

In [13]:
Xtr2, Xte2, ytr2, yte2 = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)
print(Xtr2.shape, Xte2.shape)   # (320, 48) (80, 48)

(320, 48) (80, 48)


5. Model Dasar(RBF Default)

In [14]:
base = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
base.fit(Xtr2, ytr2)

acc_base_train = accuracy_score(ytr2, base.predict(Xtr2))
acc_base_test = accuracy_score(yte2, base.predict(Xte2))
print(f'Default RBF  -> train: {acc_base_train:.4f} | test: {acc_base_test:.4f}')

Default RBF  -> train: 1.0000 | test: 1.0000


6. Hyperparameter tuning

In [15]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
param_grid = {
    'svc__C': [0.1, 1, 10, 100, 1000],
    'svc__gamma': ['scale', 0.0001, 0.001, 0.01, 0.1, 1],
}

grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1)
grid.fit(Xtr2, ytr2)

print("Parameter terbaik :", grid.best_params_)
print(f"Akurasi CV terbaik: {grid.best_score_:.4f}")

Parameter terbaik : {'svc__C': 100, 'svc__gamma': 0.001}
Akurasi CV terbaik: 1.0000


7. Evaluasi model

In [16]:
best = grid.best_estimator_
yfit = best.predict(Xte2)

acc_tuned_train = accuracy_score(ytr2, best.predict(Xtr2))
acc_tuned_test = accuracy_score(yte2, yfit)
print(f'Tuned RBF    -> train: {acc_tuned_train:.4f} | test: {acc_tuned_test:.4f}')

Tuned RBF    -> train: 1.0000 | test: 1.0000


In [17]:
from sklearn.metrics import classification_report, confusion_matrix

hasil2 = pd.DataFrame({
    "Model": ["RBF default", f"RBF tuning {grid.best_params_}"],
    "Akurasi Train": [acc_base_train, acc_tuned_train],
    "Akurasi Test": [acc_base_test, acc_tuned_test],
}).round(4)
display(hasil2)

print(classification_report(yte2, yfit, target_names=['night', 'day']))
print(confusion_matrix(yte2, yfit))

,Model,Akurasi Train,Akurasi Test
0,RBF default,1.0,1.0
1,"RBF tuning {'svc__C': 100, 'svc__gamma': 0.001}",1.0,1.0


              precision    recall  f1-score   support

       night       1.00      1.00      1.00        40
         day       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

[[40  0]
 [ 0 40]]
